# Análise de Sentimentos e Sumarização Inteligente de Avaliações de Produtos E-commerce com Transformers

**Autores**: Lara Linhares, Gustavo Teodoro, Danilo Chagas, Jose Airton, Ranulfo Mascari <br>
Departamento de Ciência da Computação (DCC) <br>
Instituto de Ciências Exatas e Tecnológicas (ICET) <br>
Universidade Federal de Lavras (UFLA)

## Objetivo

Este notebook apresenta um pipeline completo de **Processamento de Linguagem Natural (PLN)** para análise de avaliações de produtos em e-commerce brasileiro, utilizando modelos **Transformers** em estado-da-arte.

**Neste notebook nós implementamos o passo a passo para**:
- Carregar e pré-processar datasets de avaliações de e-commerce em português brasileiro
- Realizar fine-tuning do modelo **BERTimbau** para classificação de sentimentos binária
- Implementar pipeline de sumarização automática com **mBART** multilingue
- Construir um sistema completo de geração de relatórios para análise de produtos
- Avaliar modelos de classificação com métricas adequadas (Accuracy, F1, AUC-ROC)

**Referências**:
- BERTimbau: https://github.com/neuralmind-ai/portuguese-bert
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- Dataset B2W: https://www.kaggle.com/datasets/fredericods/ptbr-sentiment-analysis-datasets

**Versão**: Dezembro, 2024

## Etapa 1: Carregamento e Pré-processamento dos Dados

O pré-processamento de dados é uma etapa fundamental em qualquer pipeline de PLN. Nesta seção, carregaremos o dataset de avaliações brasileiras e prepararemos os dados para o treinamento do modelo de classificação de sentimentos.

### Dataset Escolhido: ptbr-sentiment-analysis-datasets (B2W)

Utilizaremos o dataset **ptbr-sentiment-analysis-datasets** do Kaggle, que contém avaliações reais de produtos brasileiros de fontes como B2W, Buscapé e Olist. Este dataset é ideal pois contém diversas características as quais corroboram para uma aplicação prática do nosso conhecimento adquirido durante a disciplina. As principais características do mesmo, econtram-se descritas abaixo:

| Característica | Descrição |
|----------------|-----------|
| **Fonte** | Avaliações reais de e-commerce brasileiro |
| **Língua** | Português brasileiro nativo |
| **Estrutura** | Já pré-processado com colunas `polarity` (0/1) e `rating` (1-5) |
| **Volume** | ~130k avaliações da B2W (subset principal) |
| **Disponibilidade** | Download via KaggleHub |

### Estrutura do Dataset

O dataset B2W possui as seguintes colunas principais:

- `review_text`: Texto original da avaliação escrita pelo cliente
- `review_text_processed`: Texto pré-processado (lowercase, sem acentos, tokenizado)
- `polarity`: Sentimento binário onde **0 = negativo** e **1 = positivo**
- `rating`: Classificação em estrelas de 1 a 5

### Mapeamento de Sentimentos

Para este projeto, utilizamos a classificação binária:

$$
\text{label} = \begin{cases}
0 & \text{se } \text{polarity} = 0 \text{ (Negativo)} \\
1 & \text{se } \text{polarity} = 1 \text{ (Positivo)}
\end{cases}
$$

Esta abordagem simplifica o problema de classificação multiclasse (5 estrelas) para binário, permitindo maior acurácia e interpretabilidade.

### Fundamentos Técnicos do Pré-processamento

#### Por que usar o BERTimbau?

O **BERTimbau** (neuralmind/bert-base-portuguese-cased) é um modelo BERT pré-treinado especificamente em um corpus de texto em português brasileiro. Comparado a modelos multilíngues como XLM-RoBERTa, o BERTimbau oferece:

- **Melhor performance em pt-BR**: Treinado exclusivamente em textos brasileiros
- **Vocabulário otimizado**: Tokenizador adaptado para palavras e expressões brasileiras
- **Conhecimento contextual**: Captura nuances linguísticas específicas do português

#### Tokenização e Representação

A tokenização transforma o texto em sequências de tokens (subpalavras) que o modelo pode processar:

$$
\text{Texto} \xrightarrow{\text{Tokenizer}} \text{[CLS]} + \text{tokens} + \text{[SEP]} \xrightarrow{\text{Embedding}} \mathbf{X} \in \mathbb{R}^{n \times d}
$$

Onde:
- $n$ é o número de tokens na sequência
- $d = 768$ é a dimensão dos embeddings do BERT base
- `[CLS]` é o token especial usado para classificação
- `[SEP]` marca o final da sequência

#### Parâmetros de Tokenização

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `max_length` | 256 | Reviews de e-commerce podem ser extensas |
| `truncation` | True | Garante que sequências longas sejam cortadas |
| `padding` | Dinâmico | Otimiza uso de memória no DataCollator |

#### Divisão dos Dados

Seguimos a proporção padrão para machine learning:

| Conjunto | Proporção | Finalidade |
|----------|-----------|------------|
| **Treino** | 80% | Aprendizado dos parâmetros do modelo |
| **Validação** | 10% | Tuning de hiperparâmetros e early stopping |
| **Teste** | 10% | Avaliação final imparcial do modelo |

O parâmetro `seed=42` garante reprodutibilidade dos experimentos.

### Download e Carregamento do Dataset

O código abaixo utiliza a biblioteca **KaggleHub** para fazer o download automático do dataset diretamente dos servidores do Kaggle. Após o download, carregamos o arquivo CSV do subset B2W usando **Pandas**.

Documentação KaggleHub: https://github.com/Kaggle/kagglehub

In [3]:
# Download do dataset via KaggleHub
import kagglehub

# Download da versão mais recente do dataset
path = kagglehub.dataset_download("fredericods/ptbr-sentiment-analysis-datasets")

print("Path to dataset files:", path)

# Carregamento do dataset B2W (principal)
import pandas as pd
import os

# Carregar o arquivo B2W contendo avaliações de e-commerce
b2w_path = os.path.join(path, "b2w.csv")
df = pd.read_csv(b2w_path)

# Exibir informações sobre o dataset carregado
print("Dataset carregado com sucesso!")
print(f"Tamanho do dataset: {len(df)} avaliações")
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nPrimeiras 5 linhas:")
print(df.head())

# Análise da distribuição das classes
print("\nDistribuição de polaridade (0=negativo, 1=positivo):")
print(df['polarity'].value_counts())

print("\nDistribuição de rating (1-5 estrelas):")
print(df['rating'].value_counts())

Resuming download from 588251136 bytes (331455281 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/fredericods/ptbr-sentiment-analysis-datasets?dataset_version_number=1 (588251136/919706417) bytes left.
Resuming download from https://www.kaggle.com/api/v1/datasets/download/fredericods/ptbr-sentiment-analysis-datasets?dataset_version_number=1 (588251136/919706417) bytes left.


100%|██████████| 877M/877M [00:25<00:00, 12.9MB/s]

Extracting files...


Path to dataset files: /home/teodoro/.cache/kagglehub/datasets/fredericods/ptbr-sentiment-analysis-datasets/versions/1
Dataset carregado com sucesso!
Tamanho do dataset: 132373 avaliações

Colunas disponíveis:
['original_index', 'review_text', 'review_text_processed', 'review_text_tokenized', 'polarity', 'rating', 'kfold_polarity', 'kfold_rating']

Primeiras 5 linhas:
   original_index                                        review_text  \
0           11955  Bem macio e felpudo...recomendo.  Preço imbatí...   
1           35478  Produto excepcional!  recomendo!!! inovador e ...   
2          122760  recebi o produto antes do prazo mas veio com d...   
3           17114  Bom custo beneficio. Adequado para pessoas que...   
4           19112  Além de higiênico tem o tamanho ideal. Só falt...   

                               review_text_processed  \
0  bem macio e felpudo...recomendo.  preco imbati...   
1  produto excepcional!  recomendo!!! inovador e ...   
2  recebi o produto antes do

### Conversão para Dataset Hugging Face e Divisão dos Dados

A biblioteca **Hugging Face Datasets** oferece estruturas de dados otimizadas para treinamento de modelos de PLN. O código abaixo:

1. Seleciona apenas as colunas necessárias (`text` e `label`)
2. Remove valores nulos que poderiam causar erros
3. Converte o DataFrame Pandas para o formato Dataset do Hugging Face
4. Divide os dados em conjuntos de treino, validação e teste

Documentação Datasets: https://huggingface.co/docs/datasets

In [4]:
# Converter DataFrame Pandas para Dataset do Hugging Face
from datasets import Dataset, DatasetDict

# Selecionar e renomear colunas para o formato esperado pelo modelo
df_processed = df[['review_text_processed', 'polarity']].copy()
df_processed = df_processed.rename(columns={
    'review_text_processed': 'text',  # texto de entrada
    'polarity': 'label'                # rótulo de classificação
})

# Remover valores nulos para evitar erros durante o treinamento
df_processed = df_processed.dropna()

# Garantir que os labels sejam inteiros (0 ou 1)
df_processed['label'] = df_processed['label'].astype(int)

print(f"Dataset processado: {len(df_processed)} avaliações")
print("Distribuição das classes:")
print(df_processed['label'].value_counts())

# Converter para Dataset Hugging Face
dataset = Dataset.from_pandas(df_processed)

# Dividir em train/validation/test usando proporções: 80% treino, 10% validação, 10% teste
# Primeira divisão: 80% treino, 20% restante
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)

# Segunda divisão: dividir os 20% restantes igualmente entre validação e teste
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

# Montar estrutura final do DatasetDict
dataset_final = DatasetDict({
    'train': train_test_split['train'],       # 80% dos dados
    'validation': test_val_split['train'],    # 10% dos dados  
    'test': test_val_split['test']            # 10% dos dados
})

print("\nDivisão final dos dados:")
print(f"Treino: {len(dataset_final['train'])} amostras")
print(f"Validação: {len(dataset_final['validation'])} amostras") 
print(f"Teste: {len(dataset_final['test'])} amostras")

# Exibir exemplo de avaliação
print("\nExemplo de avaliação:")
print("Texto:", dataset_final['train'][0]['text'])
print("Label:", dataset_final['train'][0]['label'])

Dataset processado: 116058 avaliações
Distribuição das classes:
label
1    80300
0    35758
Name: count, dtype: int64

Divisão final dos dados:
Treino: 92846 amostras
Validação: 11606 amostras
Teste: 11606 amostras

Exemplo de avaliação:
Texto: porque nao recomendo este produto:  1) o produto chegou bem antes do prazo. parece que isso e bom, no entanto eu entrava constantemente no site para averiguar a data da entrega, e na ultima vez, informou que atrasaria 5 dias. era para chegar no dia 01/06, depois marcou que chegaria 06/06, e no final chegou dia 18/05. qual problema disso?! eu moro em condominio de apartamento, quando chegou o produto estava viajando, ficou armazenado na portaria, causando constrangimento e desconforto aos outros condominos, alem de chover durante alguns dias, e para nao molhar tiveram que colocar junto com os porteiros.  2) o produto veio danificado: duas tabuas que fazem parte do rack, vieram amassadas, entortadas. e o painel veio lascado. a sorte que as tabuas 

### Tokenização com BERTimbau

A **tokenização** é o processo de converter texto em sequências de tokens (subpalavras) que o modelo pode processar. O tokenizador do BERTimbau utiliza o algoritmo **WordPiece**, que divide palavras desconhecidas em subpalavras conhecidas.

#### Exemplo de Tokenização

```
"Produto excelente" → ["[CLS]", "Pro", "##duto", "excel", "##ente", "[SEP]"]
```

O prefixo `##` indica que o token é uma continuação da palavra anterior.

#### Função de Tokenização

A função `tokenize_function` aplica o tokenizador a todos os exemplos do dataset:
- `truncation=True`: Corta sequências maiores que `max_length`
- `max_length=256`: Limite de tokens por sequência
- `padding=False`: Padding será aplicado dinamicamente pelo DataCollator

Documentação AutoTokenizer: https://huggingface.co/docs/transformers/main_classes/tokenizer

In [5]:
# Tokenização usando BERTimbau (modelo BERT brasileiro)
from transformers import AutoTokenizer

# Carregar tokenizador do BERTimbau - modelo BERT específico para português brasileiro
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    """
    Tokeniza o texto das avaliações usando o tokenizador do BERTimbau.
    
    Args:
        examples: Batch de exemplos contendo o campo 'text'
        
    Returns:
        Dict com input_ids, attention_mask e token_type_ids
        
    Notes:
        - Usa BERTimbau treinado especificamente em português brasileiro
        - max_length=256 para acomodar reviews mais longas
        - Padding será feito dinamicamente pelo DataCollator para eficiência
    """
    return tokenizer(
        examples["text"], 
        truncation=True,   # corta sequências maiores que max_length
        max_length=256,    # limite de tokens (reviews podem ser extensas)
        padding=False      # padding dinâmico será aplicado depois
    )

# Aplicar tokenização em todos os conjuntos de dados (batched para eficiência)
tokenized_datasets = dataset_final.map(tokenize_function, batched=True)

# Remover coluna de texto original (já temos os tokens)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Definir formato PyTorch para compatibilidade com o Trainer
tokenized_datasets.set_format("torch")

print("Colunas restantes após processamento:")
print(tokenized_datasets['train'].column_names)

# Exibir exemplo tokenizado
print("\nExemplo tokenizado:")
example = tokenized_datasets['train'][0]
print(f"Input IDs (primeiros 20): {example['input_ids'][:20]}...")
print(f"Attention Mask (primeiros 20): {example['attention_mask'][:20]}...")
print(f"Label: {example['label']}")

/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/92846 [00:00<?, ? examples/s]

Map:   0%|          | 0/11606 [00:00<?, ? examples/s]

Map:   0%|          | 0/11606 [00:00<?, ? examples/s]

Colunas restantes após processamento:
['label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']

Exemplo tokenizado:
Input IDs (primeiros 20): tensor([  101,  2113,   229, 22280,  9099, 22280,   860,  3576,   131,   205,
          114,   146,  3576,  2080,  1004,  1075,   171,  6620,   119,  4048])...
Attention Mask (primeiros 20): tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])...
Label: 0


### Data Collator e Validação Final

O **DataCollatorWithPadding** é responsável por criar batches de dados durante o treinamento. Ele aplica **padding dinâmico**, que adiciona tokens de preenchimento apenas até o tamanho da maior sequência no batch atual.

#### Vantagens do Padding Dinâmico

| Abordagem | Descrição | Eficiência |
|-----------|-----------|------------|
| Padding Fixo | Todas as sequências com mesmo tamanho (max_length) |  Desperdício de memória |
| Padding Dinâmico | Pad até o maior elemento do batch |  Uso otimizado de memória |

#### Attention Mask

A **attention mask** é um tensor binário que indica quais tokens são reais (1) e quais são padding (0):

$$
\text{attention\_mask}_i = \begin{cases}
1 & \text{se } \text{token}_i \neq \text{[PAD]} \\
0 & \text{se } \text{token}_i = \text{[PAD]}
\end{cases}
$$

Isso permite que o modelo ignore os tokens de padding durante o cálculo da atenção.

Documentação DataCollator: https://huggingface.co/docs/transformers/main_classes/data_collator

In [6]:
# Configurar Data Collator para padding dinâmico
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Data Collator configurado para padding dinâmico")

# Validação final dos dados processados
print("\nValidação dos dados processados:")
print(f"Tamanho do conjunto de treino: {len(tokenized_datasets['train'])}")
print(f"Tamanho do conjunto de validação: {len(tokenized_datasets['validation'])}")
print(f"Tamanho do conjunto de teste: {len(tokenized_datasets['test'])}")

# Verificar distribuição das classes em cada conjunto
from collections import Counter

def count_labels(split):
    """
    Conta a distribuição de labels em um split do dataset.
    
    Args:
        split: Nome do split ('train', 'validation', 'test')
        
    Returns:
        Counter com contagem de cada label
    """
    labels = [example['label'].item() for example in tokenized_datasets[split]]
    return Counter(labels)

print("\nDistribuição das classes por conjunto:")
print("Train:", dict(count_labels('train')))
print("Validation:", dict(count_labels('validation')))
print("Test:", dict(count_labels('test')))

# Demonstrar funcionamento do Data Collator
sample_batch = data_collator([tokenized_datasets['train'][i] for i in range(4)])
print(f"\nExemplo de batch após Data Collator (4 amostras):")
print(f"Input IDs shape: {sample_batch['input_ids'].shape}")
print(f"Attention Mask shape: {sample_batch['attention_mask'].shape}")
print(f"Labels shape: {sample_batch['labels'].shape}")

2025-12-05 21:36:36.875265: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 21:36:36.916504: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 21:36:37.063737: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-05 21:36:37.063786: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-05 21:36:37.064452: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2025-12-05 21:36:38.237951: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Data Collator configurado para padding dinâmico

Validação dos dados processados:
Tamanho do conjunto de treino: 92846
Tamanho do conjunto de validação: 11606
Tamanho do conjunto de teste: 11606

Distribuição das classes por conjunto:
Train: {0: 28526, 1: 64320}
Train: {0: 28526, 1: 64320}
Validation: {1: 8009, 0: 3597}
Validation: {1: 8009, 0: 3597}
Test: {0: 3635, 1: 7971}

Exemplo de batch após Data Collator (4 amostras):
Input IDs shape: torch.Size([4, 256])
Attention Mask shape: torch.Size([4, 256])
Labels shape: torch.Size([4])
Test: {0: 3635, 1: 7971}

Exemplo de batch após Data Collator (4 amostras):
Input IDs shape: torch.Size([4, 256])
Attention Mask shape: torch.Size([4, 256])
Labels shape: torch.Size([4])


## Etapa 2: Fine-tuning do Modelo para Análise de Sentimentos

O **fine-tuning** é o processo de adaptar um modelo pré-treinado para uma tarefa específica. Neste caso, adaptaremos o BERTimbau (treinado em linguagem geral) para classificação de sentimentos em avaliações de e-commerce.

### Arquitetura do Modelo

O modelo de classificação de sentimentos consiste em:

1. **Encoder BERT**: Processa o texto e gera representações contextuais
2. **Pooler**: Extrai a representação do token `[CLS]`
3. **Classification Head**: Camada linear que mapeia para as classes de saída

![Arquitetura BERT para Classificação](https://media.geeksforgeeks.org/wp-content/uploads/20250716124807294639/12.webp)

### Função de Perda: Cross-Entropy

Para classificação binária, utilizamos a **Binary Cross-Entropy Loss**:

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]
$$

Onde:
- $y_i \in \{0, 1\}$ é o label verdadeiro
- $\hat{y}_i$ é a probabilidade predita para a classe positiva
- $N$ é o número de exemplos no batch

### Por que Fine-tuning?

| Abordagem | Vantagem | Desvantagem |
|-----------|----------|-------------|
| Treinar do zero | Modelo específico | Requer muito mais dados e tempo |
| Fine-tuning | Aproveita conhecimento pré-treinado | Requer menos dados |
| Zero-shot | Sem treinamento | Performance limitada |

O fine-tuning é ideal quando temos um dataset de tamanho moderado (dezenas de milhares de exemplos).

### Carregamento do Modelo BERTimbau

O código abaixo carrega o modelo BERTimbau pré-treinado e adiciona uma **classification head** (camada de classificação) para nossa tarefa binária.

#### Parâmetros do Modelo

- `num_labels=2`: Duas classes (negativo e positivo)
- `problem_type="single_label_classification"`: Cada exemplo pertence a exatamente uma classe

O modelo possui aproximadamente **110 milhões de parâmetros**, sendo a maioria do encoder BERT pré-treinado.

Documentação AutoModelForSequenceClassification: https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification

In [7]:
# Carregamento do modelo BERTimbau para classificação binária de sentimentos
from transformers import AutoModelForSequenceClassification

model_name = "neuralmind/bert-base-portuguese-cased"

# Carregar modelo pré-treinado com head de classificação
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,  # Classificação binária: 0 (negativo) e 1 (positivo)
    problem_type="single_label_classification"
)

print("Modelo BERTimbau carregado com sucesso!")
print(f"Número de labels: {model.num_labels}")
print(f"Número total de parâmetros: {model.num_parameters():,}")

# Verificar disponibilidade de GPU para aceleração do treinamento
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDispositivo de treinamento: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória GPU disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Mover modelo para o dispositivo de treinamento
model.to(device)

/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warning

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo BERTimbau carregado com sucesso!
Número de labels: 2
Número total de parâmetros: 108,924,674

Dispositivo de treinamento: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memória GPU disponível: 6.09 GB


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Métricas de Avaliação para Classificação Binária

Para avaliar o desempenho do modelo, utilizamos um conjunto de métricas complementares:

#### Accuracy (Acurácia)

Proporção de predições corretas sobre o total:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

#### Precision (Precisão)

Dos exemplos classificados como positivos, quantos realmente são positivos:

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

#### Recall (Revocação)

Dos exemplos realmente positivos, quantos foram identificados corretamente:

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

#### F1-Score

Média harmônica entre Precision e Recall, balanceando ambas as métricas:

$$
\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

#### AUC-ROC

Área sob a curva ROC, mede a capacidade do modelo de distinguir entre classes:
- **AUC = 1.0**: Classificador perfeito
- **AUC = 0.5**: Classificador aleatório
- **AUC < 0.5**: Classificador pior que aleatório

Onde: **TP** = True Positive, **TN** = True Negative, **FP** = False Positive, **FN** = False Negative

In [8]:
# Definição das métricas de avaliação para classificação binária
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    """
    Computa métricas de avaliação para classificação binária de sentimentos.
    
    Args:
        eval_pred: Tuple contendo (predictions, labels)
            - predictions: logits do modelo (shape: [N, 2])
            - labels: labels verdadeiros (shape: [N])
            
    Returns:
        Dict com métricas: accuracy, precision, recall, f1, auc_roc
        
    Notes:
        - Usa pos_label=1 para métricas binárias (classe positiva)
        - AUC-ROC calculado com probabilidades softmax
    """
    predictions, labels = eval_pred
    
    # Extrair probabilidade da classe positiva (para AUC-ROC)
    # Aplicar softmax para converter logits em probabilidades
    preds_proba = np.exp(predictions[:, 1]) / np.exp(predictions).sum(axis=1)
    
    # Predição binária: argmax dos logits
    preds_binary = np.argmax(predictions, axis=1)
    
    # Calcular métricas principais
    accuracy = accuracy_score(labels, preds_binary)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds_binary, 
        average='binary',  # métricas para classificação binária
        pos_label=1        # classe positiva é label 1
    )
    
    # AUC-ROC mede capacidade de discriminação entre classes
    auc_roc = roc_auc_score(labels, preds_proba)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc
    }

print("Função de métricas configurada para classificação binária")
print("Métricas calculadas: accuracy, precision, recall, f1, auc_roc")

Função de métricas configurada para classificação binária
Métricas calculadas: accuracy, precision, recall, f1, auc_roc


### Configuração dos Hiperparâmetros de Treinamento

Os **TrainingArguments** definem todos os hiperparâmetros do processo de fine-tuning. A escolha cuidadosa desses parâmetros é crucial para o sucesso do treinamento.

#### Hiperparâmetros Principais

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `learning_rate` | 2e-5 | Taxa baixa para preservar conhecimento pré-treinado |
| `batch_size` | 16 | Compromisso entre memória GPU e estabilidade |
| `num_train_epochs` | 3 | Suficiente para fine-tuning sem overfitting |
| `weight_decay` | 0.01 | Regularização L2 para evitar overfitting |
| `fp16` | True | Mixed precision para acelerar treinamento |

#### Learning Rate para Fine-tuning

Para fine-tuning de modelos BERT, recomenda-se usar learning rates muito baixas (1e-5 a 5e-5) para não "esquecer" o conhecimento pré-treinado. Este fenômeno é conhecido como **catastrophic forgetting**.

#### Estratégia de Avaliação e Salvamento

- `evaluation_strategy="epoch"`: Avaliar modelo ao final de cada época
- `save_strategy="epoch"`: Salvar checkpoint ao final de cada época
- `load_best_model_at_end=True`: Carregar o melhor modelo ao final do treinamento
- `metric_for_best_model="f1"`: Usar F1-score para selecionar o melhor modelo

Documentação TrainingArguments: https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments

In [9]:
# Configuração dos argumentos de treinamento (hiperparâmetros)
from transformers import TrainingArguments

output_dir = "./results/sentiment_model"  # diretório para salvar checkpoints

training_args = TrainingArguments(
    # Diretórios de saída
    output_dir=output_dir,
    logging_dir='./logs',  # diretório para logs do TensorBoard
    
    # Hiperparâmetros de otimização
    learning_rate=2e-5,    # taxa de aprendizado baixa para fine-tuning
    weight_decay=0.01,     # regularização L2 para prevenir overfitting
    
    # Configuração de batch
    per_device_train_batch_size=16,  # batch size por GPU no treino
    per_device_eval_batch_size=16,   # batch size por GPU na avaliação
    
    # Épocas e logging
    num_train_epochs=3,    # 3 épocas é suficiente para fine-tuning
    logging_steps=100,     # log a cada 100 steps
    
    # Estratégia de avaliação e salvamento
    evaluation_strategy="epoch",     # avaliar ao final de cada época
    save_strategy="epoch",           # salvar checkpoint a cada época
    load_best_model_at_end=True,     # carregar melhor modelo ao final
    metric_for_best_model="f1",      # usar F1 como métrica principal
    greater_is_better=True,          # maior F1 = melhor modelo
    
    # Otimizações de performance
    fp16=True,                       # mixed precision (acelera treinamento)
    dataloader_pin_memory=False,     # compatibilidade com alguns sistemas
)

print("Training Arguments configurados:")
print(f"- Learning rate: {training_args.learning_rate}")
print(f"- Batch size (treino): {training_args.per_device_train_batch_size}")
print(f"- Batch size (avaliação): {training_args.per_device_eval_batch_size}")
print(f"- Número de épocas: {training_args.num_train_epochs}")
print(f"- Weight decay: {training_args.weight_decay}")
print(f"- Mixed precision (fp16): {training_args.fp16}")
print(f"- Métrica para melhor modelo: {training_args.metric_for_best_model}")
print(f"- Output dir: {training_args.output_dir}")

Training Arguments configurados:
- Learning rate: 2e-05
- Batch size (treino): 16
- Batch size (avaliação): 16
- Número de épocas: 3
- Weight decay: 0.01
- Mixed precision (fp16): True
- Métrica para melhor modelo: f1
- Output dir: ./results/sentiment_model


### Treinamento com o Trainer

O **Trainer** é a classe principal do Hugging Face que abstrai todo o loop de treinamento. Ele gerencia automaticamente:

- Forward e backward pass
- Otimização dos parâmetros
- Avaliação periódica
- Salvamento de checkpoints
- Logging de métricas
- Gerenciamento de memória GPU

#### Processo de Treinamento

1. **Forward Pass**: O modelo processa um batch de dados
2. **Cálculo da Loss**: Compara predições com labels verdadeiros
3. **Backward Pass**: Calcula gradientes via backpropagation
4. **Atualização de Parâmetros**: Optimizer ajusta os pesos do modelo

Este processo se repete para todos os batches em cada época.

Documentação Trainer: https://huggingface.co/docs/transformers/main_classes/trainer

In [10]:
# Instanciação e execução do Trainer
from transformers import Trainer

# Configurar o Trainer com modelo, dados e hiperparâmetros
trainer = Trainer(
    model=model,                                    # modelo BERTimbau carregado
    args=training_args,                             # hiperparâmetros de treinamento
    train_dataset=tokenized_datasets["train"],      # dataset de treino tokenizado
    eval_dataset=tokenized_datasets["validation"],  # dataset de validação tokenizado
    data_collator=data_collator,                    # collator para padding dinâmico
    compute_metrics=compute_metrics,                # função de métricas customizada
)

print("Trainer configurado com sucesso!")
print(f"Exemplos de treino: {len(tokenized_datasets['train'])}")
print(f"Exemplos de validação: {len(tokenized_datasets['validation'])}")

# Iniciar o processo de fine-tuning
print("\n" + "="*50)
print("INICIANDO TREINAMENTO")
print("="*50)

train_result = trainer.train()

# Exibir resultados do treinamento
print("\n" + "="*50)
print("TREINAMENTO CONCLUÍDO!")
print("="*50)
print(f"Tempo total: {train_result.metrics['train_runtime']:.2f} segundos")
print(f"Tempo por época: {train_result.metrics['train_runtime']/training_args.num_train_epochs:.2f} segundos")
print(f"Exemplos processados por segundo: {train_result.metrics['train_samples_per_second']:.2f}")
print(f"Steps totais: {train_result.metrics.get('train_steps', 'N/A')}")

Trainer configurado com sucesso!
Exemplos de treino: 92846
Exemplos de validação: 11606

INICIANDO TREINAMENTO


  0%|          | 0/17409 [00:00<?, ?it/s]

{'loss': 0.2762, 'grad_norm': 3.785270929336548, 'learning_rate': 1.988741455568959e-05, 'epoch': 0.02}
{'loss': 0.1703, 'grad_norm': 7.3162078857421875, 'learning_rate': 1.977253144925039e-05, 'epoch': 0.03}
{'loss': 0.1703, 'grad_norm': 7.3162078857421875, 'learning_rate': 1.977253144925039e-05, 'epoch': 0.03}
{'loss': 0.1567, 'grad_norm': 19.031856536865234, 'learning_rate': 1.9657648342811192e-05, 'epoch': 0.05}
{'loss': 0.1567, 'grad_norm': 19.031856536865234, 'learning_rate': 1.9657648342811192e-05, 'epoch': 0.05}
{'loss': 0.1504, 'grad_norm': 4.516191482543945, 'learning_rate': 1.954276523637199e-05, 'epoch': 0.07}
{'loss': 0.1504, 'grad_norm': 4.516191482543945, 'learning_rate': 1.954276523637199e-05, 'epoch': 0.07}
{'loss': 0.129, 'grad_norm': 0.3665864169597626, 'learning_rate': 1.9427882129932796e-05, 'epoch': 0.09}
{'loss': 0.129, 'grad_norm': 0.3665864169597626, 'learning_rate': 1.9427882129932796e-05, 'epoch': 0.09}
{'loss': 0.1416, 'grad_norm': 4.3003644943237305, 'learn

  0%|          | 0/726 [00:00<?, ?it/s]

{'eval_loss': 0.10521728545427322, 'eval_accuracy': 0.9692400482509047, 'eval_precision': 0.9845491388044579, 'eval_recall': 0.9706580097390436, 'eval_f1': 0.9775542282301163, 'eval_auc_roc': 0.9923543582277278, 'eval_runtime': 35.5951, 'eval_samples_per_second': 326.056, 'eval_steps_per_second': 20.396, 'epoch': 1.0}
{'loss': 0.0801, 'grad_norm': 0.13600148260593414, 'learning_rate': 1.322764087540927e-05, 'epoch': 1.02}
{'loss': 0.0801, 'grad_norm': 0.13600148260593414, 'learning_rate': 1.322764087540927e-05, 'epoch': 1.02}
{'loss': 0.0863, 'grad_norm': 1.7114665508270264, 'learning_rate': 1.3112757768970074e-05, 'epoch': 1.03}
{'loss': 0.0863, 'grad_norm': 1.7114665508270264, 'learning_rate': 1.3112757768970074e-05, 'epoch': 1.03}
{'loss': 0.0796, 'grad_norm': 15.816426277160645, 'learning_rate': 1.2997874662530876e-05, 'epoch': 1.05}
{'loss': 0.0796, 'grad_norm': 15.816426277160645, 'learning_rate': 1.2997874662530876e-05, 'epoch': 1.05}
{'loss': 0.0973, 'grad_norm': 0.203827619552

  0%|          | 0/726 [00:00<?, ?it/s]

{'eval_loss': 0.11068756878376007, 'eval_accuracy': 0.9690677235912459, 'eval_precision': 0.9827107521453812, 'eval_recall': 0.9722811836683731, 'eval_f1': 0.9774681478692023, 'eval_auc_roc': 0.9931510189763233, 'eval_runtime': 35.5435, 'eval_samples_per_second': 326.529, 'eval_steps_per_second': 20.426, 'epoch': 2.0}
{'loss': 0.0442, 'grad_norm': 1.495684266090393, 'learning_rate': 6.5667183640645645e-06, 'epoch': 2.02}
{'loss': 0.0442, 'grad_norm': 1.495684266090393, 'learning_rate': 6.5667183640645645e-06, 'epoch': 2.02}
{'loss': 0.0528, 'grad_norm': 4.225092887878418, 'learning_rate': 6.451835257625366e-06, 'epoch': 2.03}
{'loss': 0.0528, 'grad_norm': 4.225092887878418, 'learning_rate': 6.451835257625366e-06, 'epoch': 2.03}
{'loss': 0.0478, 'grad_norm': 0.02855965867638588, 'learning_rate': 6.336952151186169e-06, 'epoch': 2.05}
{'loss': 0.0478, 'grad_norm': 0.02855965867638588, 'learning_rate': 6.336952151186169e-06, 'epoch': 2.05}
{'loss': 0.0325, 'grad_norm': 0.25376176834106445,

  0%|          | 0/726 [00:00<?, ?it/s]

{'eval_loss': 0.14819000661373138, 'eval_accuracy': 0.9692400482509047, 'eval_precision': 0.9798093804865814, 'eval_recall': 0.9755275315270321, 'eval_f1': 0.9776637677532378, 'eval_auc_roc': 0.9920039392714056, 'eval_runtime': 35.4054, 'eval_samples_per_second': 327.803, 'eval_steps_per_second': 20.505, 'epoch': 3.0}
{'train_runtime': 3816.5164, 'train_samples_per_second': 72.982, 'train_steps_per_second': 4.561, 'train_loss': 0.0818503490983036, 'epoch': 3.0}

TREINAMENTO CONCLUÍDO!
Tempo total: 3816.52 segundos
Tempo por época: 1272.17 segundos
Exemplos processados por segundo: 72.98
Steps totais: N/A
{'train_runtime': 3816.5164, 'train_samples_per_second': 72.982, 'train_steps_per_second': 4.561, 'train_loss': 0.0818503490983036, 'epoch': 3.0}

TREINAMENTO CONCLUÍDO!
Tempo total: 3816.52 segundos
Tempo por época: 1272.17 segundos
Exemplos processados por segundo: 72.98
Steps totais: N/A


### Avaliação no Conjunto de Teste

Após o treinamento, avaliamos o modelo no **conjunto de teste**, que contém dados nunca vistos durante o treinamento ou validação. Esta avaliação fornece uma estimativa imparcial do desempenho do modelo em dados reais.

#### Importância do Conjunto de Teste

- **Imparcialidade**: Dados não usados para ajustar hiperparâmetros
- **Generalização**: Mede capacidade de generalizar para novos dados
- **Comparação**: Permite comparar diferentes modelos de forma justa

### Interpretação dos Resultados

#### Guia de Interpretação das Métricas

| F1-Score | Interpretação | Ação Recomendada |
|----------|---------------|------------------|
| > 0.90 | Excelente | Modelo pronto para produção |
| 0.80 - 0.90 | Muito bom | Pode ser usado com monitoramento |
| 0.70 - 0.80 | Bom | Considerar melhorias |
| 0.60 - 0.70 | Aceitável | Revisar dados e hiperparâmetros |
| < 0.60 | Insuficiente | Reavaliação necessária |

#### AUC-ROC

| AUC-ROC | Interpretação |
|---------|---------------|
| > 0.90 | Excelente discriminação entre classes |
| 0.80 - 0.90 | Boa discriminação |
| 0.70 - 0.80 | Discriminação aceitável |
| 0.50 - 0.70 | Discriminação fraca |
| = 0.50 | Modelo aleatório |


In [11]:
# Avaliação final do modelo no conjunto de teste
print("Avaliando modelo no conjunto de teste...")

test_results = trainer.evaluate(tokenized_datasets["test"])

# Exibir resultados formatados
print("\nRESULTADOS NO CONJUNTO DE TESTE:")
print("-"*40)
for metric, value in test_results.items():
    if metric.startswith('eval_'):
        metric_name = metric.replace('eval_', '').upper()
        print(f"{metric_name}: {value:.4f}")

# Salvar o melhor modelo para uso posterior
model_save_path = "./best_sentiment_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("\n" + "="*50)
print(f"Modelo salvo em: {model_save_path}")
print("Arquivos salvos: config.json, model.safetensors, tokenizer files")
print("="*50)

Avaliando modelo no conjunto de teste...


  0%|          | 0/726 [00:00<?, ?it/s]


RESULTADOS NO CONJUNTO DE TESTE:
----------------------------------------
LOSS: 0.1420
ACCURACY: 0.9702
PRECISION: 0.9794
RECALL: 0.9772
F1: 0.9783
AUC_ROC: 0.9931
RUNTIME: 35.2400
SAMPLES_PER_SECOND: 329.3410
STEPS_PER_SECOND: 20.6020

Modelo salvo em: ./best_sentiment_model
Arquivos salvos: config.json, model.safetensors, tokenizer files

Modelo salvo em: ./best_sentiment_model
Arquivos salvos: config.json, model.safetensors, tokenizer files


### Teste Manual do Modelo Treinado

Para validar qualitativamente o modelo, realizamos testes manuais com avaliações de exemplo. Isso permite verificar se o modelo captura corretamente o sentimento em diferentes contextos.

#### Pipeline de Inferência

O **pipeline** do Hugging Face simplifica o processo de inferência, encapsulando:
1. Tokenização do texto de entrada
2. Forward pass pelo modelo
3. Conversão de logits para probabilidades
4. Retorno da predição com score de confiança

Documentação Pipeline: https://huggingface.co/docs/transformers/main_classes/pipelines

In [12]:
# Carregar modelo treinado usando pipeline para facilitar inferência
from transformers import pipeline

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="./best_sentiment_model",
    tokenizer="./best_sentiment_model",
    device=0 if torch.cuda.is_available() else -1  # usar GPU se disponível
)

# Avaliações de exemplo para teste qualitativo
test_reviews = [
    "Produto excelente, chegou antes do prazo e funciona perfeitamente!",
    "Péssimo atendimento, produto defeituoso e entrega atrasada.",
    "O produto é bom, mas o preço está um pouco alto.",
    "Recomendo muito, qualidade excepcional!",
    "Não comprem, veio com defeito e não consigo trocar."
]

print("TESTES MANUAIS DO MODELO TREINADO")
print("="*60)

for review in test_reviews:
    result = sentiment_classifier(review)
    
    # Converter label do modelo para texto legível
    sentiment = "Positivo" if result[0]['label'] == 'LABEL_1' else "Negativo"
    confidence = result[0]['score']
    
    print(f"\nReview: \"{review[:60]}...\"" if len(review) > 60 else f"\nReview: \"{review}\"")
    print(f"Predição: {sentiment}")
    print(f"Confiança: {confidence:.1%}")
    print("-" * 60)

TESTES MANUAIS DO MODELO TREINADO

Review: "Produto excelente, chegou antes do prazo e funciona perfeita..."
Predição: Positivo
Confiança: 100.0%
------------------------------------------------------------

Review: "Péssimo atendimento, produto defeituoso e entrega atrasada."
Predição: Negativo
Confiança: 99.6%
------------------------------------------------------------

Review: "O produto é bom, mas o preço está um pouco alto."
Predição: Positivo
Confiança: 98.9%
------------------------------------------------------------

Review: "Recomendo muito, qualidade excepcional!"
Predição: Positivo
Confiança: 100.0%
------------------------------------------------------------

Review: "Não comprem, veio com defeito e não consigo trocar."
Predição: Negativo
Confiança: 100.0%
------------------------------------------------------------


## Etapa 3: Sumarização de Avaliações e Geração de Relatórios

Nesta etapa, construiremos um sistema completo que combina **classificação de sentimentos** com **sumarização automática** para gerar relatórios executivos sobre avaliações de produtos.

### Arquitetura do Sistema

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   Avaliações    │────▶│   Classificador  │────▶│   Agrupamento   │
│   (Texto Raw)   │     │   (BERTimbau)    │     │   por Produto   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                                                          │         
                                                          ▼         
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│    Relatório    │◀────│   Sumarizador    │◀────│  Pontos +/-     │
│    Executivo    │     │     (mBART)      │     │   Agregados     │
└─────────────────┘     └──────────────────┘     └─────────────────┘
```

### Modelo de Sumarização: mBART

O **mBART** (Multilingual BART) é um modelo Transformer sequence-to-sequence treinado em 50 idiomas, incluindo português. Características:

| Aspecto | Descrição |
|---------|-----------|
| **Arquitetura** | Encoder-Decoder Transformer |
| **Pré-treinamento** | Denoising autoencoder multilingue |
| **Idiomas** | 50 idiomas, incluindo pt-BR |
| **Parâmetros** | ~610 milhões |
| **Uso** | Sumarização, tradução, geração de texto |

### Aplicações em Engenharia de Dados

Este sistema demonstra um pipeline completo de **ETL com NLP**:

- **Extract**: Coleta de avaliações de múltiplas fontes
- **Transform**: Classificação de sentimentos + sumarização
- **Load**: Relatórios estruturados para consumo downstream

Documentação mBART: https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt

### Sistema de Relatórios de E-commerce

**Arquitetura do Sistema**:

1. **Classificação de Sentimentos**: Modelo BERTimbau analisa cada avaliação
2. **Agrupamento por Produto**: Estatísticas agregadas por categoria
3. **Sumarização Inteligente**: mBART condensa pontos positivos/negativos
4. **Relatório Estruturado**: Formato legível para gestores

**Aplicações em Engenharia de Dados**:

**Extração**: Pipeline automatizado processa milhares de avaliações

**Transformação**: 
- Classificação de sentimentos
- Agregação por produto/categoria
- Geração de métricas de satisfação

**Loading/Consumo**:
- **APIs**: Relatórios servidos via REST APIs
- **Data Warehouse**: Métricas armazenadas para business intelligence
- **Dashboards**: Power BI/Tableau consomem dados em tempo real
- **Message Queue**: Alertas automáticos via Kafka para produtos com baixa satisfação

**Governança de Dados**:
- **Qualidade**: Validação automática de dados de entrada
- **Conformidade**: Anonimização de dados pessoais
- **Auditoria**: Logs de processamento para rastreabilidade

**Valor de Negócio**:
- **Tempo de Resposta**: De dias para minutos na identificação de problemas
- **Ações Proativas**: Alertas automáticos para produtos defeituosos
- **ROI Mensurável**: Redução de custos com suporte e aumento de vendas

In [13]:
# Carregamento do modelo de sumarização mBART multilingue
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Nome do modelo multilingue para sumarização
model_name = "facebook/mbart-large-50-many-to-many-mmt"

# Verificar dispositivo disponível (GPU acelera significativamente)
device_idx = 0 if torch.cuda.is_available() else -1
device_str = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Carregando modelo mBART no dispositivo: {device_str}")
print("Isso pode levar alguns minutos...")

# Carregar tokenizer e modelo separadamente para maior controle
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Mover modelo para GPU se disponível
model.to(device_str)

# Configurar idioma de origem no tokenizer (crucial para pt-BR)
tokenizer.src_lang = "pt_XX"  # código do português no mBART

# Obter ID do token para forçar geração em português
forced_bos_token_id = tokenizer.lang_code_to_id["pt_XX"]
model.config.forced_bos_token_id = forced_bos_token_id

# 6. Obter o ID do token para o idioma de destino
# Este ID será usado para forçar a geração em Português
forced_bos_token_id = tokenizer.lang_code_to_id["pt_XX"]

# Criar o pipeline, passando o modelo e o tokenizer já configurados
summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device=device_idx,
)

print("\nModelo de sumarização mBART carregado!")
print(f"Dispositivo: {'GPU' if device_idx == 0 else 'CPU'}")
print(f"Parâmetros do modelo: ~610 milhões")

# Teste rápido do sumarizador
test_text = "Este produto é excelente, chegou rápido e funciona perfeitamente. Recomendo muito!"
summary = summarizer(
    test_text,
    max_length=50,   # tamanho máximo do resumo
    min_length=20,   # tamanho mínimo do resumo
    do_sample=False, # geração determinística
    forced_bos_token_id=forced_bos_token_id  # garantir saída em português
)[0]['summary_text']

print(f"\nTeste de sumarização:")
print(f"Original: {test_text}")
print(f"Resumo: {summary}")

Carregando modelo mBART no dispositivo: cuda:0
Isso pode levar alguns minutos...


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Your max_length is set to 50, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)



Modelo de sumarização mBART carregado!
Dispositivo: GPU
Parâmetros do modelo: ~610 milhões


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/transformers/generation/utils.py:1339: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(



Teste de sumarização:
Original: Este produto é excelente, chegou rápido e funciona perfeitamente. Recomendo muito!
Resumo: Este produto é excelente, chegou muito rápido e funciona perfeitamente. Eu recomendo muito! ""


### Carregamento do Modelo de Sentimentos Treinado

Agora carregamos o modelo BERTimbau que treinamos na etapa anterior para classificar o sentimento de cada avaliação. Este modelo será usado em conjunto com o sumarizador para gerar relatórios completos.

In [14]:
# Carregar modelo de sentimentos treinado anteriormente
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="./best_sentiment_model",
    tokenizer="./best_sentiment_model", 
    device=device_str,
    max_length=512,    # limite de tokens para textos longos
    truncation=True    # truncar textos que excedam o limite
)

print("Modelo de sentimentos carregado!")
print(f"Dispositivo: {device_str}")

# Validar funcionamento do classificador
test_reviews = [
    "Produto maravilhoso, entrega rápida!",
    "Péssimo, veio com defeito.",
    "Mais ou menos, preço justo."
]

print("\nValidação do classificador de sentimentos:")
print("-" * 50)
for review in test_reviews:
    result = sentiment_classifier(review)
    label = "Positivo" if result[0]['label'] == 'LABEL_1' else "Negativo"
    score = result[0]['score']
    print(f"'{review}'")
    print(f"   → {label} (confiança: {score:.1%})")

Modelo de sentimentos carregado!
Dispositivo: cuda:0

Validação do classificador de sentimentos:
--------------------------------------------------
'Produto maravilhoso, entrega rápida!'
   → Positivo (confiança: 100.0%)
'Péssimo, veio com defeito.'
   → Negativo (confiança: 100.0%)
'Mais ou menos, preço justo.'
   → Positivo (confiança: 99.4%)


### Sistema de Análise e Geração de Relatórios

O código abaixo implementa a classe `ProductReviewAnalyzer`, que encapsula toda a lógica de análise de avaliações e geração de relatórios.

#### Funcionalidades da Classe

1. **`analyze_reviews()`**: Processa todas as avaliações, classificando sentimentos e agrupando por produto
2. **`generate_product_report()`**: Gera relatório detalhado para um produto específico

#### Estrutura do Relatório

O relatório gerado inclui:
- **Estatísticas gerais**: Total de avaliações, distribuição positivo/negativo
- **Pontos positivos**: Resumo das avaliações positivas (sumarizado)
- **Pontos de melhoria**: Resumo das avaliações negativas (sumarizado)
- **Recomendações**: Ações sugeridas baseadas nos dados
- **Exemplos**: Amostra de avaliações reais

#### Padrão de Design

A classe segue o princípio de **Single Responsibility**: cada método tem uma única responsabilidade bem definida, facilitando manutenção e testes.

In [15]:
# Sistema de análise de avaliações e geração de relatórios por produto
import pandas as pd
from collections import defaultdict

class ProductReviewAnalyzer:
    """
    Sistema para análise de sentimentos em avaliações de produtos e 
    geração de relatórios executivos automatizados.
    
    Attributes:
        sentiment_model: Pipeline de classificação de sentimentos
        summarizer: Pipeline de sumarização de texto
        
    Methods:
        analyze_reviews: Processa avaliações e agrupa por produto
        generate_product_report: Gera relatório detalhado para um produto
    """
    
    def __init__(self, sentiment_model, summarizer_model):
        """
        Inicializa o analisador com os modelos de NLP.
        
        Args:
            sentiment_model: Pipeline de classificação de sentimentos
            summarizer_model: Pipeline de sumarização de texto
        """
        self.sentiment_model = sentiment_model
        self.summarizer = summarizer_model
    
    def analyze_reviews(self, reviews_df, product_column='product_category', 
                       text_column='review_text_processed'):
        """
        Analisa todas as avaliações e retorna estatísticas agregadas.
        
        Args:
            reviews_df: DataFrame com as avaliações
            product_column: Nome da coluna de agrupamento (produto/categoria)
            text_column: Nome da coluna com o texto da avaliação
            
        Returns:
            Dict com estatísticas por produto/categoria
        """
        # Estrutura para armazenar resultados agregados
        results = defaultdict(lambda: {
            'total_reviews': 0,
            'positive_reviews': [],
            'negative_reviews': [],
            'positive_count': 0,
            'negative_count': 0,
            'sentiment_distribution': {'positive': 0, 'negative': 0}
        })
        
        print(f"Analisando {len(reviews_df)} avaliações...")
        
        for idx, row in reviews_df.iterrows():
            product = row.get(product_column, 'unknown')
            text = row[text_column]
            
            # Classificar sentimento usando modelo treinado
            sentiment_result = self.sentiment_model(text)
            sentiment = 'positive' if sentiment_result[0]['label'] == 'LABEL_1' else 'negative'
            
            # Agregar resultados
            results[product]['total_reviews'] += 1
            results[product]['sentiment_distribution'][sentiment] += 1
            
            if sentiment == 'positive':
                results[product]['positive_reviews'].append(text)
                results[product]['positive_count'] += 1
            else:
                results[product]['negative_reviews'].append(text)
                results[product]['negative_count'] += 1
            
            # Log de progresso a cada 1000 avaliações
            if (idx + 1) % 1000 == 0:
                print(f"Processadas {idx + 1} avaliações...")
        
        return dict(results)
    
    def generate_product_report(self, product_name, analysis_results, max_reviews_for_summary=5):
        """
        Gera relatório executivo detalhado para um produto específico.
        
        Args:
            product_name: Nome/ID do produto para gerar relatório
            analysis_results: Dict com resultados da análise
            max_reviews_for_summary: Número máximo de reviews para sumarização
            
        Returns:
            String formatada em Markdown com o relatório completo
        """
        if product_name not in analysis_results:
            return f"Produto '{product_name}' não encontrado nos dados."
        
        data = analysis_results[product_name]
        total = data['total_reviews']
        positive_pct = (data['positive_count'] / total) * 100 if total > 0 else 0
        negative_pct = (data['negative_count'] / total) * 100 if total > 0 else 0
        
        # Construir relatório em formato Markdown
        report = f"""
# Relatório de Análise de Sentimentos - {product_name}

## Estatísticas Gerais
- **Total de avaliações**: {total}
- **Avaliações positivas**: {data['positive_count']} ({positive_pct:.1f}%)
- **Avaliações negativas**: {data['negative_count']} ({negative_pct:.1f}%)

## Pontos Positivos
"""
        
        # Sumarizar avaliações positivas
        if data['positive_reviews']:
            positive_sample = data['positive_reviews'][:max_reviews_for_summary]
            positive_text = " ".join(positive_sample)
            
            try:
                positive_summary = self.summarizer(
                    positive_text, 
                    max_length=100, 
                    min_length=30, 
                    do_sample=False
                )[0]['summary_text']
                report += f"{positive_summary}\n"
            except Exception as e:
                report += f"Não foi possível gerar resumo dos pontos positivos. Erro: {e}\n"
        
        report += "\n## Pontos de Melhoria\n"
        
        # Sumarizar avaliações negativas
        if data['negative_reviews']:
            negative_sample = data['negative_reviews'][:max_reviews_for_summary]
            negative_text = " ".join(negative_sample)
            
            try:
                negative_summary = self.summarizer(
                    negative_text, 
                    max_length=100, 
                    min_length=30, 
                    do_sample=False
                )[0]['summary_text']
                report += f"{negative_summary}\n"
            except Exception as e:
                report += f"Não foi possível gerar resumo dos pontos negativos. Erro: {e}\n"
        
        # Recomendações baseadas nos dados
        report += "\n## Recomendações\n"
        if positive_pct > 70:
            report += "- **Produto bem avaliado**: Manter qualidade e padrões de entrega\n"
        elif positive_pct > 50:
            report += "- **Produto aceitável**: Focar em melhorar pontos negativos identificados\n"
        else:
            report += "- **Atenção necessária**: Revisar qualidade do produto e processo de entrega\n"
        
        # Exemplos de avaliações
        report += "\n## Exemplos de Avaliações\n"
        
        report += "\n**Avaliações Positivas:**\n"
        for i, review in enumerate(data['positive_reviews'][:3], 1):
            report += f"{i}. \"{review[:100]}...\"\n"
        
        report += "\n**Avaliações Negativas:**\n"
        for i, review in enumerate(data['negative_reviews'][:3], 1):
            report += f"{i}. \"{review[:100]}...\"\n"
        
        return report

# Instanciar o analisador com os modelos carregados
analyzer = ProductReviewAnalyzer(sentiment_classifier, summarizer)
print("Sistema de análise de produtos configurado!")
print("Métodos disponíveis: analyze_reviews(), generate_product_report()")

Sistema de análise de produtos configurado!
Métodos disponíveis: analyze_reviews(), generate_product_report()


### Execução da Análise Completa

Agora executaremos a análise em uma amostra do dataset. Por questões de tempo de processamento, utilizamos uma amostra de 5000 avaliações. Em produção, o sistema processaria todo o dataset.

#### Observações sobre Performance

- **Tempo de processamento**: ~1-2 segundos por avaliação (com GPU)
- **Escalabilidade**: Para grandes volumes, considerar processamento em batch
- **Otimização**: Uso de GPU acelera significativamente a inferência

In [16]:
# Executar análise em uma amostra do dataset
# Usar amostra para demonstração (ajustar conforme recursos disponíveis)
sample_size = min(5000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)

print(f"Iniciando análise de {sample_size} avaliações...")
print("="*50)

# Executar análise completa
# Nota: Usando 'rating' como coluna de agrupamento (o dataset não tem categoria de produto)
analysis_results = analyzer.analyze_reviews(
    df_sample, 
    product_column='rating',              # agrupar por rating (1-5 estrelas)
    text_column='review_text_processed'   # coluna com texto processado
)

print("Análise concluída!")
print(f"Produtos analisados: {len(analysis_results)}")

# Calcular estatísticas gerais
total_reviews = sum(data['total_reviews'] for data in analysis_results.values())
total_positive = sum(data['positive_count'] for data in analysis_results.values())
total_negative = sum(data['negative_count'] for data in analysis_results.values())

print("\n" + "="*50)
print("ESTATÍSTICAS GERAIS")
print("="*50)
print(f"Total de avaliações analisadas: {total_reviews}")
print(f"Avaliações positivas: {total_positive} ({(total_positive/total_reviews)*100:.1f}%)")
print(f"Avaliações negativas: {total_negative} ({(total_negative/total_reviews)*100:.1f}%)")

# Mostrar distribuição por rating
print("\nDistribuição por Rating:")
print("-"*40)
sorted_products = sorted(analysis_results.items(), 
                        key=lambda x: x[1]['total_reviews'], 
                        reverse=True)

for rating, data in sorted_products:
    pct_positive = (data['positive_count']/data['total_reviews'])*100 if data['total_reviews'] > 0 else 0
    print(f"Rating {rating}: {data['total_reviews']} avaliações ({pct_positive:.0f}% positivas)")

Iniciando análise de 5000 avaliações...
Analisando 5000 avaliações...


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Processadas 90000 avaliações...
Processadas 11000 avaliações...
Processadas 84000 avaliações...
Processadas 84000 avaliações...
Processadas 33000 avaliações...
Processadas 33000 avaliações...
Processadas 41000 avaliações...
Processadas 41000 avaliações...
Processadas 49000 avaliações...
Processadas 49000 avaliações...
Processadas 128000 avaliações...
Processadas 128000 avaliações...
Análise concluída!
Produtos analisados: 5

ESTATÍSTICAS GERAIS
Total de avaliações analisadas: 5000
Avaliações positivas: 3445 (68.9%)
Avaliações negativas: 1555 (31.1%)

Distribuição por Rating:
----------------------------------------
Rating 5: 1810 avaliações (99% positivas)
Rating 4: 1235 avaliações (99% positivas)
Rating 1: 1059 avaliações (1% positivas)
Rating 3: 588 avaliações (71% positivas)
Rating 2: 308 avaliações (2% positivas)
Análise concluída!
Produtos analisados: 5

ESTATÍSTICAS GERAIS
Total de avaliações analisadas: 5000
Avaliações positivas: 3445 (68.9%)
Avaliações negativas: 1555 (31.1%)



### Geração de Relatório Executivo

Finalmente, geramos um relatório completo em formato Markdown para o grupo com mais avaliações. Este relatório poderia ser:

- Exportado como PDF para stakeholders
- Integrado em dashboards de BI
- Enviado automaticamente via email
- Armazenado em data lake para análise histórica

In [17]:
# Gerar relatório detalhado para o produto/rating mais avaliado
if sorted_products:
    # Identificar o rating com mais avaliações
    top_rating = sorted_products[0][0]
    top_rating_str = str(top_rating)
    
    print(f"Gerando relatório para avaliações com Rating {top_rating_str}...")
    print("="*50)
    
    # Gerar relatório completo
    report = analyzer.generate_product_report(top_rating, analysis_results)
    
    # Salvar relatório em arquivo Markdown
    report_filename = f'report_rating_{top_rating_str.replace(" ", "_")}.md'
    with open(report_filename, 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(f"Relatório salvo em: {report_filename}")
    
    # Exibir preview do relatório
    print("\n" + "="*60)
    print("PREVIEW DO RELATÓRIO")
    print("="*60)
    print(report)
    print("="*60)
else:
    print("Nenhum dado disponível para gerar relatório.")

Gerando relatório para avaliações com Rating 5...
Relatório salvo em: report_rating_5.md

PREVIEW DO RELATÓRIO

# Relatório de Análise de Sentimentos - 5

## Estatísticas Gerais
- **Total de avaliações**: 1810
- **Avaliações positivas**: 1794 (99.1%)
- **Avaliações negativas**: 16 (0.9%)

## Pontos Positivos
Gostava do produto. Easy de usar. Alcanceu muito rápido e bem resistente. Ótimo serviço e entrega no prazo, e o produto de excelente qualidade, e o pré também esta otimo. Eu gostava do aparelho, processador muito rápido, câmara top, recomendo. Comprei por recommendá-lo, agora eu também recomendo. Presentei ao meu marido e ele adoro!! e rápido, cheio de engueis, a câmara e otimo!

## Pontos de Melhoria
a entrega foi incompleta, pedi 40 unidades e foram entregues 24. as 16 unidades que faltam deram-me um prazo de 7 dias porem, não recebi. ao voltar a falar, passei o prazo de mais 7 dias, ou seja, 21 dias para a entrega total do pedido. absurdo! nem no site e nem nos telefones da Amér

---

# Conclusão

Este notebook apresentou um pipeline completo de **Processamento de Linguagem Natural** para análise de avaliações de e-commerce brasileiro utilizando modelos Transformer estado-da-arte.

## O que foi Aprendido

1. **Pré-processamento de Dados**: Carregamento, limpeza e tokenização de dados textuais para modelos BERT
2. **Fine-tuning de Modelos**: Adaptação do BERTimbau para classificação binária de sentimentos
3. **Avaliação de Modelos**: Uso de métricas adequadas (Accuracy, F1, Precision, Recall, AUC-ROC)
4. **Sumarização Automática**: Aplicação do mBART para geração de resumos em português
5. **Sistema de Relatórios**: Construção de pipeline para geração automatizada de insights

## Resultados Obtidos

- Modelo de classificação de sentimentos treinado com performance adequada para produção
- Sistema de análise capaz de processar milhares de avaliações automaticamente
- Relatórios executivos gerados automaticamente com sumarização inteligente

---

**Tecnologias Utilizadas**: Python, PyTorch, Hugging Face Transformers, BERTimbau, mBART, Pandas, Scikit-learn

---

### Fine-tuning ptt5-base para relatórios em PT
Implementa o pipeline do plano: gerar pares JSON→Markdown a partir das análises atuais, preparar tokenização, opcionalmente treinar (`RUN_FINETUNE`) e expor um gerador para substituir o sumarizador antigo (com toggle).

In [18]:
# Configuração do gerador ptt5-base e helpers de linearização
from pathlib import Path
import json
import random
from typing import List, Dict, Any

MODEL_REPORT_BASE = "unicamp-dl/ptt5-base-portuguese-vocab"
REPORT_MODEL_DIR = Path("./report-model-ft")
REPORT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Flags de execução
RUN_FINETUNE = False  # defina True para treinar; mantém False para não disparar treino pesado por engano
USE_LORA = False      # habilite se quiser treinar adapters leves (requer peft)
MAX_SOURCE_LENGTH = 640
MAX_TARGET_LENGTH = 320


def _truncate_list(texts: List[str], max_items: int = 3, max_chars: int = 200) -> List[str]:
    """Limita quantidade e tamanho de exemplos para evitar estouro de contexto."""
    return [t[:max_chars] for t in texts[:max_items]]


def linearize_payload(product_name: str, stats: Dict[str, Any], max_examples: int = 3) -> str:
    """Converte o payload agregado em texto plano compacto para o ptt5."""
    pos = _truncate_list(stats.get("positive_reviews", []), max_examples)
    neg = _truncate_list(stats.get("negative_reviews", []), max_examples)
    return (
        f"produto: {product_name} | total_avaliacoes: {stats.get('total_reviews', 0)} | "
        f"positivas: {stats.get('positive_count', 0)} | negativas: {stats.get('negative_count', 0)} | "
        f"exemplos_positivos: {' || '.join(pos) if pos else 'N/D'} | "
        f"exemplos_negativos: {' || '.join(neg) if neg else 'N/D'}"
    )


def build_report_pairs(analysis_results: Dict[str, Any], analyzer, max_products: int = 30) -> List[Dict[str, str]]:
    """Gera pares sintéticos usando o relatório atual como alvo (seed dataset)."""
    pairs = []
    for product, stats in list(analysis_results.items())[:max_products]:
        try:
            input_text = linearize_payload(product, stats)
            target_md = analyzer.generate_product_report(product, analysis_results)
            pairs.append({"input_text": input_text.strip(), "target_text": target_md.strip()})
        except Exception as exc:  # evita quebrar em casos anômalos
            print(f"Aviso: falha ao gerar par para {product}: {exc}")
            continue
    random.shuffle(pairs)
    return pairs


report_pairs = build_report_pairs(analysis_results, analyzer, max_products=30)
print(f"Pares gerados para fine-tune (seed): {len(report_pairs)}")
if report_pairs:
    print("Exemplo de entrada:", report_pairs[0]["input_text"][:240], "...")
    print("Exemplo de alvo:", report_pairs[0]["target_text"][:240], "...")


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Pares gerados para fine-tune (seed): 5
Exemplo de entrada: produto: 5 | total_avaliacoes: 1810 | positivas: 1794 | negativas: 16 | exemplos_positivos: gostamos muito do produto!. facil manuseio. alcanca bastante velocidade e e bem resistente. || atendimento otimo e entrega no prazo,  e o produto de ...
Exemplo de alvo: # Relatório de Análise de Sentimentos - 5

## Estatísticas Gerais
- **Total de avaliações**: 1810
- **Avaliações positivas**: 1794 (99.1%)
- **Avaliações negativas**: 16 (0.9%)

## Pontos Positivos
Gostava do produto. Easy de usar. Alcanceu ...


In [19]:
# Tokenização e preparação do dataset seq2seq
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

if not report_pairs:
    print("Nenhum par disponível; gere analysis_results primeiro ou aumente max_products.")
else:
    report_dataset = Dataset.from_list(report_pairs)
    report_dataset = report_dataset.train_test_split(test_size=0.1, seed=42)

    report_tokenizer = AutoTokenizer.from_pretrained(MODEL_REPORT_BASE)

    def preprocess_reports(batch):
        model_inputs = report_tokenizer(batch["input_text"],
                                         max_length=MAX_SOURCE_LENGTH,
                                         truncation=True)
        labels = report_tokenizer(batch["target_text"],
                                   max_length=MAX_TARGET_LENGTH,
                                   truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_report_ds = report_dataset.map(
        preprocess_reports,
        batched=True,
        remove_columns=report_dataset["train"].column_names
    )

    data_collator = DataCollatorForSeq2Seq(report_tokenizer, padding=True)
    print(tokenized_report_ds)


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})


In [20]:
# Treinador Seq2Seq (LoRA opcional) — não executa se RUN_FINETUNE=False
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments

if report_pairs:
    if USE_LORA:
        try:
            from peft import LoraConfig, get_peft_model
        except ImportError:
            raise ImportError("peft não instalado; defina USE_LORA=False ou instale peft.")

    model_report = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPORT_BASE)

    if USE_LORA:
        lora_cfg = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q", "k", "v", "o"]
        )
        model_report = get_peft_model(model_report, lora_cfg)
        print("LoRA habilitado para ptt5-base.")

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(REPORT_MODEL_DIR),
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_ratio=0.08,
        label_smoothing_factor=0.1,
        predict_with_generate=True,
        evaluation_strategy="steps",
        eval_steps=200,
        save_steps=200,
        logging_steps=50,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to=[]
    )

    trainer = Seq2SeqTrainer(
        model=model_report,
        args=training_args,
        train_dataset=tokenized_report_ds["train"],
        eval_dataset=tokenized_report_ds["test"],
        tokenizer=report_tokenizer,
        data_collator=data_collator
    )

    if RUN_FINETUNE:
        train_result = trainer.train()
        trainer.save_model(REPORT_MODEL_DIR)
        report_tokenizer.save_pretrained(REPORT_MODEL_DIR)
        print("Treino concluído e modelo salvo em", REPORT_MODEL_DIR)
    else:
        print("RUN_FINETUNE=False → pulei a etapa de treino. Ajuste a flag para treinar.")


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

RUN_FINETUNE=False → pulei a etapa de treino. Ajuste a flag para treinar.


In [30]:
# Inferência com o modelo fine-tunado (ou base, se ainda não treinado)
from transformers import pipeline

report_generator = None
FORCE_CPU = False  # defina False se tiver VRAM suficiente


def load_report_generator(model_dir: Path = REPORT_MODEL_DIR):
    """Carrega o gerador; se não houver checkpoint salvo, usa o modelo base."""
    global report_generator
    if report_generator is None:
        has_checkpoint = model_dir.exists() and (model_dir / "config.json").exists()
        model_path = str(model_dir if has_checkpoint else MODEL_REPORT_BASE)
        device_to_use = -1 if FORCE_CPU or (not torch.cuda.is_available()) else device_idx
        report_generator = pipeline(
            "text2text-generation",
            model=model_path,
            tokenizer=model_path,
            device=device_to_use,
        )
    return report_generator


def generate_report_llm(product_name: str, stats: Dict[str, Any]):
    """Gera relatório em Markdown usando o gerador ptt5-base (fine-tunado se disponível)."""
    generator = load_report_generator()
    input_text = linearize_payload(product_name, stats)
    output = generator(
        input_text,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        temperature=0.3
    )[0]["generated_text"]
    return output


# Exemplo rápido (usa o primeiro produto do analysis_results)
if analysis_results:
    sample_product, sample_stats = next(iter(analysis_results.items()))
    demo_report = generate_report_llm(sample_product, sample_stats)
    print("Pré-visualização do relatório gerado (LLM):\n", demo_report[:500], "...")
else:
    print("analysis_results vazio: rode a análise antes de gerar relatórios.")


/home/teodoro/Documents/Estudos/Faculdade/DeepLearning/pln/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 12.00 MiB is free. Including non-PyTorch memory, this process has 5.63 GiB memory in use. Of the allocated memory 5.27 GiB is allocated by PyTorch, and 250.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [24]:
print(demo_report[:1000])  # exibe os primeiros 1000 caracteres do relatório

produto: 1 | total_avaliacoes: 1059 | positivas: 9 | negativas: 1050 | exemplos_positivos: parece que comprei esta roupa em um bazar de esquina. || quero agradecer as americanas pelo belo servico de entrega. compreii dia 20/04 dia do lancamento, o pagamento foi aprovado no dia seguinte, a nota fiscal foi emitida dia 24/04 e saiu para o transporte || temperatura nao cai a menos de 22°c indicados no visor digital, quando deveria ser de 12 a 18°c...,.. | exemplos­negativos: a imagem do anuncio traz um case rigido preto (entregue) e uma capa emborrachada cor laranja para o hd (nao entregue). ao informar que a capa nao foi entregue, recebi uma ligacao informando que eu est || produto simplesmente nao chegou. estou no aguardo para que entrem em contato. | | tive que mandar ele para a assistencia 3 vezes, deu inumeros problemas de varios tipos, seja de tela azul da morte a problemas de carcaca! pessimo notebook!


### Geração estruturada com LangChain (prompt few-shot)
Vamos usar o mesmo modelo ptt5 (base ou fine-tunado) via LangChain, com prompt estruturado e poucos exemplos para melhorar a coerência e o formato do relatório.

In [ ]:
# LangChain (LCEL) + HuggingFacePipeline para relatório estruturado (usa dados agregados de fato)
try:
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_huggingface import HuggingFacePipeline
    from transformers import pipeline as hf_pipeline_cls
except ImportError as e:
    raise ImportError(
        "Instale langchain>=0.2, langchain-core e langchain-huggingface. "
        "Exemplo: pip install 'langchain>=0.2' langchain-core langchain-huggingface"
    ) from e


def build_lc_pipeline():
    """Cria pipeline HF com parâmetros de geração explícitos para o LangChain."""
    has_checkpoint = REPORT_MODEL_DIR.exists() and (REPORT_MODEL_DIR / "config.json").exists()
    model_path = str(REPORT_MODEL_DIR if has_checkpoint else MODEL_REPORT_BASE)
    return hf_pipeline_cls(
        "text2text-generation",
        model=model_path,
        tokenizer=model_path,
        max_new_tokens=MAX_TARGET_LENGTH,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        temperature=0.3,
    )


lc_pipe = build_lc_pipeline()
hf_llm = HuggingFacePipeline(pipeline=lc_pipe)

fewshot_examples = """
### Exemplo
Entrada (contexto):
- produto: Fone ABC
- total_avaliacoes: 120
- positivas: 90
- negativas: 30
- exemplos_positivos: som ótimo || bateria dura
- exemplos_negativos: quebrou em 1 mês || assistência lenta
Saída:
# Relatório do Produto: Fone ABC
- Estatísticas: 120 avaliações; 75.0% positivas; 25.0% negativas.
- Pontos positivos: som nítido; bateria elogiada.
- Pontos negativos: relatos de quebra precoce e assistência demorada.
- Recomendações: checar controle de qualidade do lote; reforçar suporte pós-venda.
"""

prompt = PromptTemplate.from_template(
    "Você é um analista que gera relatórios concisos em Markdown sobre produtos. "
    "Use apenas os dados fornecidos; se algo faltar, escreva 'N/D'.\n"
    "Formato esperado:\n"
    "# Relatório do Produto: <nome>\n"
    "- Estatísticas: <total> avaliações; <pct_pos>% positivas; <pct_neg>% negativas.\n"
    "- Pontos positivos: ...\n"
    "- Pontos negativos: ...\n"
    "- Recomendações: ...\n\n"
    f"{fewshot_examples}\n"
    "### Nova Entrada (contexto real)\n"
    "- produto: {produto}\n"
    "- total_avaliacoes: {total}\n"
    "- positivas: {positivas}\n"
    "- negativas: {negativas}\n"
    "- exemplos_positivos: {exemplos_pos}\n"
    "- exemplos_negativos: {exemplos_neg}\n"
    "Saída:\n"
)

chain = prompt | hf_llm | StrOutputParser()


def generate_report_langchain(product_name: str, stats: Dict[str, Any]):
    total = stats.get("total_reviews", 0)
    pos = stats.get("positive_count", 0)
    neg = stats.get("negative_count", 0)
    exemplos_pos = " || ".join(_truncate_list(stats.get("positive_reviews", []), 3)) or "N/D"
    exemplos_neg = " || ".join(_truncate_list(stats.get("negative_reviews", []), 3)) or "N/D"
    return chain.invoke({
        "produto": product_name,
        "total": total,
        "positivas": pos,
        "negativas": neg,
        "exemplos_pos": exemplos_pos,
        "exemplos_neg": exemplos_neg,
    }).strip()


# Demonstração (usa o primeiro produto já analisado)
if analysis_results:
    sample_product, sample_stats = next(iter(analysis_results.items()))
    lc_report = generate_report_langchain(sample_product, sample_stats)
    print(lc_report)
else:
    print("analysis_results vazio: rode a análise antes de gerar relatórios.")


Você é um analista que gera relatórios concisos em Markdown sobre produtos. Use apenas os dados fornecidos; se algo faltar, escreva 'N/D'. Formato esperado: # Relatório do Produto: <nome> - Estatísticas:  Basqueteboltotal> avaliações; <pct_pos>% positivas; Мpct__neg>% negativas. - Pontos positivos: ... - Ponto, negativos: ...- Recomendações: ... ### Exemplo Entrada (contexto): - produto: Fone ABC - total_avaliacoes: 120 - positivas: 90 - negativas: 30 - exemplos_positivos: som ótimo || bateria dura - exemplos negativos: quebrou em 1 mês || assistência lenta Saída: # relatório do Produto, FoneBC - EstatísticaS: 120 avaliações; 75.0% positivas); 25.0 % negativas.- Pontos positivas: som nítido; bateria elogiada. -pontos negativos: relatos de quebra precoce e assistência demorada. - Recom recomendações: checar controle de qualidade do lote; reforçar suporte pós-venda. ## # Nova Entrada(contextO real) - produto : 1 - total.avaliaCoes: 1059 - positivas): 9 - negativas): 1050 - exemplos­posit

In [29]:
# Gerar um relatório com LangChain e salvar em arquivo
from pathlib import Path

REPORT_OUT = Path("reports/report-langchain.md")
REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)

if not analysis_results:
    raise ValueError("analysis_results está vazio. Rode a análise antes de gerar o relatório.")

sample_product, sample_stats = next(iter(analysis_results.items()))
report_text = generate_report_langchain(sample_product, sample_stats)

with open(REPORT_OUT, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Relatório salvo em: {REPORT_OUT}")
print(report_text[:800] + ("..." if len(report_text) > 800 else ""))


Relatório salvo em: reports/report-langchain.md
Você é um analista que gera relatórios concisos em Markdown sobre produtos. Use apenas os dados fornecidos; se algo faltar, escreva 'N/D'. Formato esperado: # Relatório do Produto: <nome> - Estatísticas:  Basqueteboltotal> avaliações; <pct_pos>% positivas; Мpct__neg>% negativas. - Pontos positivos: ... - Ponto, negativos: ...- Recomendações: ... ### Exemplo Entrada (contexto): - produto: Fone ABC - total_avaliacoes: 120 - positivas: 90 - negativas: 30 - exemplos_positivos: som ótimo || bateria dura - exemplos negativos: quebrou em 1 mês || assistência lenta Saída: # relatório do Produto, FoneBC - EstatísticaS: 120 avaliações; 75.0% positivas); 25.0 % negativas.- Pontos positivas: som nítido; bateria elogiada. -pontos negativos: relatos de quebra precoce e assistência demorada. - Recom reco...
